In [1]:
# =========================================================
# FAVAR Connectedness Baseline
# - PCA-based latent factors
# - VAR on [observed variables + latent factors]
# - Generalized FEVD / connectedness
# =========================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.api import VAR

# -----------------------------
# Config
# -----------------------------
DATA_PATH = "./spillover_data.csv"
OUT_DIR = "./output_favar"

DATE_COL = "Date"
OBS_VARS = ["dlog_SOLVPN", "dlog_COPPER", "dlog_DXY", "d_UST10Y", "dlog_VIX"]

N_FACTORS = 2
P = 1
H = 10

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df[[DATE_COL] + OBS_VARS].dropna().reset_index(drop=True)

# -----------------------------
# Factor extraction via PCA
# -----------------------------
scaler = StandardScaler()
X_std = scaler.fit_transform(df[OBS_VARS])

pca = PCA(n_components=N_FACTORS, random_state=42)
factors = pca.fit_transform(X_std)

factor_cols = [f"Factor{i+1}" for i in range(N_FACTORS)]
factor_df = pd.DataFrame(factors, columns=factor_cols, index=df.index)

favar_df = pd.concat([df[[DATE_COL]], df[OBS_VARS], factor_df], axis=1)
ALL_VARS = OBS_VARS + factor_cols

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative explained:", pca.explained_variance_ratio_.cumsum())

# -----------------------------
# Estimate FAVAR
# -----------------------------
model = VAR(favar_df[ALL_VARS])
res = model.fit(P)

print(res.summary())

# -----------------------------
# Generalized FEVD helper
# -----------------------------
def generalized_fevd_from_var_results(res, H):
    """
    res: statsmodels VARResults
    Returns row-normalized generalized FEVD matrix
    """
    Sigma = res.sigma_u.values
    A_list = [res.coefs[i] for i in range(res.k_ar)]  # each (k,k)
    k = Sigma.shape[0]
    p = len(A_list)

    def var_companion(A_list):
        p = len(A_list)
        k = A_list[0].shape[0]
        top = np.hstack(A_list)
        if p == 1:
            return top
        bottom = np.hstack([np.eye(k*(p-1)), np.zeros((k*(p-1), k))])
        return np.vstack([top, bottom])

    F = var_companion(A_list)
    kp = k*p if p > 1 else k
    J = np.hstack([np.eye(k), np.zeros((k, kp-k))]) if p > 1 else np.eye(k)

    Phi = []
    Fpow = np.eye(kp)
    for h in range(H):
        Phi.append(J @ Fpow @ J.T)
        Fpow = Fpow @ F

    theta = np.zeros((k, k))
    sigdiag = np.diag(Sigma).copy()
    sigdiag[sigdiag <= 1e-12] = 1e-12

    for i in range(k):
        for j in range(k):
            e_i = np.zeros((k,1)); e_i[i,0] = 1
            e_j = np.zeros((k,1)); e_j[j,0] = 1
            num = 0.0
            den = 0.0
            for h in range(H):
                ph = Phi[h]
                num += float((e_i.T @ ph @ Sigma @ e_j) ** 2 / sigdiag[j])
                den += float(e_i.T @ ph @ Sigma @ ph.T @ e_i)
            theta[i, j] = num / max(den, 1e-12)

    row_sum = theta.sum(axis=1, keepdims=True)
    row_sum[row_sum <= 1e-12] = 1e-12
    return theta / row_sum

def spillover_from_fevd(theta):
    k = theta.shape[0]
    total = 100.0 * (theta.sum() - np.trace(theta)) / k
    to_ = 100.0 * (theta.sum(axis=0) - np.diag(theta))
    from_ = 100.0 * (theta.sum(axis=1) - np.diag(theta))
    net_ = to_ - from_
    return total, to_, from_, net_

theta = generalized_fevd_from_var_results(res, H)
total, to_, from_, net_ = spillover_from_fevd(theta)

# -----------------------------
# Save matrices / summary
# -----------------------------
theta_df = pd.DataFrame(theta, index=ALL_VARS, columns=ALL_VARS)
theta_df.to_csv(os.path.join(OUT_DIR, "favar_fevd_matrix.csv"))

summary_rows = []
for i, v in enumerate(ALL_VARS):
    summary_rows.append({
        "Variable": v,
        "TO": to_[i],
        "FROM": from_[i],
        "NET": net_[i]
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUT_DIR, "favar_connectedness_summary.csv"), index=False)

# observed-only submatrix
obs_theta_df = theta_df.loc[OBS_VARS, OBS_VARS]
obs_theta_df.to_csv(os.path.join(OUT_DIR, "favar_observed_only_fevd_matrix.csv"))

meta = {
    "model": "FAVAR Connectedness (PCA factors)",
    "observed_vars": OBS_VARS,
    "factor_cols": factor_cols,
    "n_factors": N_FACTORS,
    "lag_order": P,
    "fevd_horizon": H,
    "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "cumulative_explained_variance": pca.explained_variance_ratio_.cumsum().tolist(),
    "total_spillover": float(total)
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

# -----------------------------
# Plots
# -----------------------------
plt.figure(figsize=(7, 5))
plt.imshow(theta_df, aspect="auto")
plt.xticks(range(len(ALL_VARS)), ALL_VARS, rotation=45, ha="right")
plt.yticks(range(len(ALL_VARS)), ALL_VARS)
plt.title("FAVAR FEVD Matrix")
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "favar_fevd_heatmap.png"), dpi=300)
plt.show()

print("Total spillover:", total)
print(summary_df)
print("Saved to:", OUT_DIR)

FileNotFoundError: [Errno 2] No such file or directory: './spillover_data.csv'